In [9]:
from pathlib import Path
from typing import List, Union, Optional
from astropy.table import Table
import astropy.io.fits as pf
from numpy import concatenate, diff, sqrt, full, median, array
from uncertainties import ufloat

In [14]:
def read_tess_spoc(tic: int,
                   datadir: Union[Path, str],
                   sectors: Optional[Union[List[int], str]] = 'all',
                   use_pdc: bool = False,
                   remove_contamination: bool = True):

    def file_filter(f, partial_tic, sectors):
        _, sector, tic, _, _ = f.name.split('-')
        if sectors != 'all':
            return int(sector[1:]) in sectors and str(partial_tic) in tic
        else:
            return str(partial_tic) in tic

    files = [f for f in sorted(Path(datadir).glob('tess*_dvt.fits')) if file_filter(f, tic, sectors)]
    fcol = 'PDCSAP_FLUX' if use_pdc else 'SAP_FLUX'
    times, fluxes, sectors = [], [], []
    for f in files:
        tb = Table.read(f)
        bjdrefi = tb.meta['BJDREFI']
        df = tb.to_pandas().dropna(subset=['TIME', 'SAP_FLUX', 'PDCSAP_FLUX'])
        times.append(df['TIME'].values.copy() + bjdrefi)
        fluxes.append(array(df[fcol].values, 'd'))
        fluxes[-1] /= median(fluxes[-1])
        if use_pdc and not remove_contamination:
            contamination = 1 - tb.meta['CROWDSAP']
            fluxes[-1] = contamination + (1 - contamination) * fluxes[-1]
        sectors.append(full(fluxes[-1].size, pf.getval(f, 'sector')))

    return (concatenate(times), concatenate(fluxes), concatenate(sectors),
            [diff(f).std() / sqrt(2) for f in fluxes])

In [20]:
def file_filter(f, partial_tic, sectors):
        _, sector, tic, _, _ = f.name.split('-')
        if sectors != 'all':
            return int(sector[1:]) in sectors and str(partial_tic) in tic
        else:
            return str(partial_tic) in tic

In [15]:
npop         = 30
mcmc_repeats = 4
datadir = 'data/WASP-44b/mastDownload/TESS/tess2018263035959-s0003-0000000012862099-0123-s/'
tic = '12862099'
zero_epoch = ufloat(2455434.37600,4e-4)
period = ufloat(2.4238039,8.7e-6)

In [16]:
read_tess_spoc(tic=tic,datadir=datadir,sectors='all')

ValueError: need at least one array to concatenate

In [24]:
sorted(Path(datadir).glob('tess*_dvt.fits'))

[PosixPath('data/WASP-44b/mastDownload/TESS/tess2018263035959-s0003-0000000012862099-0123-s/tess2018263124740-s0003-s0003-0000000012862099-00405_dvt.fits'),
 PosixPath('data/WASP-44b/mastDownload/TESS/tess2018263035959-s0003-0000000012862099-0123-s/tess2018267104341-s0003-s0003-0000000012862099-00126_dvt.fits')]